In [1]:
%load_ext autoreload 
%autoreload 2

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
os.chdir(os.getenv("PROJECT_ROOT"))

In [3]:
from src.clients import connect_mysql, connect_mongodb, connect_neo4j

In [4]:
conn = connect_mongodb()

In [5]:
objs = conn['raw_game_dates'].find().to_list()

In [6]:
import pandas as pd
df = pd.DataFrame(objs)

,_id,GAME_ID,GAME_DATE,HOME_TEAM,AWAY_TEAM,SEASON_LABEL,SEASON_TYPE,created_at,updated_at,jhash
0,69d63ec7f4fe3ca28e9b44bf,0021901318,2020-08-14,TOR,DEN,2019-20,Regular_Season,2026-04-08T11:40:55.978422+00:00,2026-04-08T11:40:55.978428+00:00,fd7b126b58c2c085fbc656f9956aec013925d02aad05e6...
1,69d63ec7f4fe3ca28e9b44c0,0021901317,2020-08-14,LAC,OKC,2019-20,Regular_Season,2026-04-08T11:40:55.978464+00:00,2026-04-08T11:40:55.978465+00:00,e2133a8345239a88bc28f596980eef346371e1aa0d5e72...
2,69d63ec7f4fe3ca28e9b44c1,0021901316,2020-08-14,IND,MIA,2019-20,Regular_Season,2026-04-08T11:40:55.978473+00:00,2026-04-08T11:40:55.978474+00:00,3ec67a37d2854656c37ae6da26885a3863c52142b6e2ba...
3,69d63ec7f4fe3ca28e9b44c2,0021901316,2020-08-14,IND,MIA,2019-20,Regular_Season,2026-04-08T11:40:55.978479+00:00,2026-04-08T11:40:55.978480+00:00,3ec67a37d2854656c37ae6da26885a3863c52142b6e2ba...
4,69d63ec7f4fe3ca28e9b44c3,0021901315,2020-08-14,HOU,PHI,2019-20,Regular_Season,2026-04-08T11:40:55.978484+00:00,2026-04-08T11:40:55.978486+00:00,0fbe5fc8d91256efdf7a78543e32220e31d8992b606e22...
...,...,...,...,...,...,...,...,...,...,...
2271,69d63f52f4fe3ca28ea7f42f,0040600121,2007-04-21,TOR,NJN,2006-07,Playoffs,2026-04-08T11:43:14.262050+00:00,2026-04-08T11:43:14.262052+00:00,be088b004434d7619698567350d22515b1c293899b19eb...
2272,69d63f52f4fe3ca28ea7f430,0040600171,2007-04-21,HOU,UTA,2006-07,Playoffs,2026-04-08T11:43:14.262056+00:00,2026-04-08T11:43:14.262057+00:00,34bcd285feeae1a815667603d714fdd706505d4afb43df...
2273,69d63f52f4fe3ca28ea7f431,0040600171,2007-04-21,HOU,UTA,2006-07,Playoffs,2026-04-08T11:43:14.262062+00:00,2026-04-08T11:43:14.262063+00:00,34bcd285feeae1a815667603d714fdd706505d4afb43df...
2274,69d63f52f4fe3ca28ea7f432,0040600131,2007-04-21,CHI,MIA,2006-07,Playoffs,2026-04-08T11:43:14.262068+00:00,2026-04-08T11:43:14.262069+00:00,df6eecf99007285007c55dea6bb0d0ce8b04ef95b3abbb...


In [14]:
df.isna().sum(axis=0)

_id             0
GAME_ID         0
GAME_DATE       0
HOME_TEAM       0
AWAY_TEAM       0
SEASON_LABEL    0
SEASON_TYPE     0
created_at      0
updated_at      0
jhash           0
dtype: int64

In [4]:
conn = connect_mysql()
cursor = conn.cursor()

In [5]:
cursor.execute("SHOW TABLES;")

In [6]:
tables = cursor.fetchall()

In [7]:
table_names, *_ = zip(*tables)

In [8]:
table_names

('agg_player_season_totals',
 'agg_team_game_totals',
 'dim_date',
 'dim_game',
 'dim_player',
 'dim_position',
 'dim_team',
 'fact_player_game_stats')

In [9]:
GET_COUNT = "SELECT COUNT(*) FROM {table};"

for table in table_names:
    cursor.execute(GET_COUNT.format(table=table))
    count = cursor.fetchall()

    print(f"{table}: {count} rows")

agg_player_season_totals: [(0,)] rows
agg_team_game_totals: [(0,)] rows
dim_date: [(47,)] rows
dim_game: [(55,)] rows
dim_player: [(574,)] rows
dim_position: [(6,)] rows
dim_team: [(28,)] rows
fact_player_game_stats: [(2000,)] rows


In [11]:
conn.close()

In [12]:
conn = connect_neo4j()

In [28]:
with open('sql/neo4j/003_analytics.cypher', 'r') as f:
    queries = f.read()

query = queries.split(";")[2]
with conn.session() as session:
    result = session.run(query)
    df = result.to_df()

ClientError: {neo4j_code: Neo.ClientError.Statement.ParameterMissing} {message: Expected parameter(s): sourceTeamId} {gql_status: 50N42} {gql_status_description: error: general processing exception - unexpected error. Unexpected error has occurred. See debug log for details.}

In [33]:
conn.close()

In [29]:
df

,seasonLabel,sourcePersonId,playerName,sourceTeamId,teamName,gamesPlayed,totalPoints,totalAssists,totalRebounds,avgPoints,avgAssists,avgRebounds
0,2000-01,711,Jerry Stackhouse,1610612765,Pistons,5,198,24,19,39.60,4.80,3.80
1,2000-01,1503,Tracy McGrady,1610612753,Magic,6,150,18,40,25.00,3.00,6.67
2,2000-01,951,Ray Allen,1610612749,Bucks,5,131,37,25,26.20,7.40,5.00
3,2000-01,397,Reggie Miller,1610612754,Pacers,6,131,22,19,21.83,3.67,3.17
4,2000-01,949,Shareef Abdur-Rahim,1610612763,Grizzlies,5,129,12,42,25.80,2.40,8.40
...,...,...,...,...,...,...,...,...,...,...,...,...
593,2000-01,166,Ron Harper,1610612747,Lakers,2,0,2,3,0.00,1.00,1.50
594,2000-01,2068,Lavor Postell,1610612752,Knicks,1,0,0,1,0.00,0.00,1.00
595,2000-01,895,Tyrone Corbin,1610612761,Raptors,1,0,0,0,0.00,0.00,0.00
596,2000-01,1917,Wang Zhi-zhi,1610612742,Mavericks,1,0,0,0,0.00,0.00,0.00
